# Solar Filament Segmentation Challenge 2026 — v2: Fine-Scale Structures

v1 (U-Net ResNet34 at 1024, plain threshold) reached **val PQ 0.41 / public 0.36**.
Error analysis showed the masks covered the dark filament core but **lost the barbs**: the thin, thread-like features that the competition lists as its first challenge. Thin masks lower IoU (SQ) and push matches below the 0.5 IoU cut-off (RQ).

v2 is built around preserving fine structure:
| Problem | Change |
|---|---|
| Barbs erased by downsampling 2048 → 1024 | Train at **native 2048** resolution on random **1024 crops** (biased toward filaments); predict on the full 2048 image, so there's no resizing anywhere |
| Faint, low-contrast barbs | 3-channel input: **raw**, **CLAHE** (local contrast), **black-hat** (highlights thin dark structures narrower than the kernel) |
| Loss ignores thin structures | **BCE + Dice + clDice**: clDice compares soft skeletons, so missing a thin barb or breaking a filament costs as much as missing a thick part |
| Model capacity | **ResNet50** encoder + **scSE attention** decoder, 40 epochs |
| Annotator disagreement | Target = **soft consensus** (mean of all annotators' masks per image) |
| Faint barb pixels cut by a hard threshold; fragmentation | **Hysteresis thresholding**: grow low-threshold regions from high-confidence seeds |
| Holes inside predictions | **Fill holes** (ground-truth polygons are solid) |
| Prediction noise | **8x dihedral TTA** |
| Diagnosis | Validation reports **PQ, SQ, RQ, TP/FP/FN** |

## 1. Setup

In [ ]:
!pip install -q segmentation-models-pytorch pycocotools kagglehub python-dotenv

In [ ]:
import os, json, random, time
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy import ndimage
from pycocotools import mask as mask_utils
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm

CFG = dict(
    crop=1024,            # training crop size at native 2048 resolution
    encoder='resnet50',
    batch_size=4,
    epochs=40,
    lr=3e-4,
    repeats=2,            # random crops per training image per epoch
    fg_crop_prob=0.7,     # fraction of crops centred on a filament pixel
    blackhat_k=21,        # black-hat kernel: highlights dark structures thinner than this (px)
    cldice_w=0.5,         # weight of the clDice (skeleton) loss
    skel_iters=10,        # soft-skeleton iterations
    tta=8,                # 1, 2 (h-flip) or 8 (dihedral)
    val_frac=0.15,
    seed=42,
    num_workers=0 if os.name == 'nt' else 2,
)
FULL = 2048
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = DEVICE == 'cuda'

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(CFG['seed'])
print('Device:', DEVICE)

## 2. Download the dataset
On Kaggle, add the competition data via **+ Add Input** (it appears under `/kaggle/input`).
Locally, `kagglehub` downloads it using `KAGGLE_API_TOKEN` from the environment / `.env`.

In [ ]:
COMP = 'filament-segmentation-2026'
ANN_NAME = 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'

def locate_dataset(root):
    hits = list(Path(root).rglob(ANN_NAME))
    return hits[0].parent.parent if hits else None

DATA_ROOT = locate_dataset('/kaggle/input') if Path('/kaggle/input').exists() else None
if DATA_ROOT is None:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    import kagglehub
    DATA_ROOT = locate_dataset(kagglehub.competition_download(COMP))

assert DATA_ROOT is not None, 'Dataset not found - did you accept the competition rules?'
ANN_PATH = next((DATA_ROOT / 'train').rglob(ANN_NAME))
print('Dataset root:', DATA_ROOT)

## 3. Load annotations

In [ ]:
IMG_EXTS = ('.jpg', '.jpeg', '.png')

def index_images(d):
    return {p.stem: p for p in Path(d).rglob('*') if p.suffix.lower() in IMG_EXTS}

train_paths = index_images(DATA_ROOT / 'train')
test_paths = index_images(DATA_ROOT / 'test')
print(f'train images: {len(train_paths)}  test images: {len(test_paths)}')

with open(ANN_PATH) as f:
    coco = json.load(f)

anns_by_img = defaultdict(list)
for a in coco['annotations']:
    anns_by_img[str(a['image_id'])].append(a)

# One sample per (annotator, image) pair
samples = []
for im in coco['images']:
    stem = Path(im['file_name']).stem
    anns = anns_by_img.get(str(im['id']), [])
    if stem in train_paths and anns:
        samples.append(dict(sample_id=str(im['id']), stem=stem, path=train_paths[stem],
                            h=im.get('height', FULL), w=im.get('width', FULL), anns=anns))

samples_by_stem = defaultdict(list)
for s in samples:
    samples_by_stem[s['stem']].append(s)

print(f'{len(samples)} annotator samples over {len(samples_by_stem)} unique images')
assert samples, 'No annotations matched images - check image_id / file_name fields'

In [ ]:
def ann_to_mask(ann, h, w):
    seg = ann['segmentation']
    if isinstance(seg, list):
        if seg and isinstance(seg[0], (int, float)):   # flat polygon
            seg = [seg]
        rle = mask_utils.merge(mask_utils.frPyObjects(seg, h, w))
    elif isinstance(seg.get('counts'), list):          # uncompressed RLE
        rle = mask_utils.frPyObjects(seg, h, w)
    else:                                              # compressed RLE
        rle = seg
    return mask_utils.decode(rle).astype(bool)

def instance_masks(s):
    return [ann_to_mask(a, s['h'], s['w']) for a in s['anns']]

def semantic_mask(s):
    m = np.zeros((s['h'], s['w']), np.uint8)
    for im in instance_masks(s):
        m[im] = 1
    return m

def read_gray(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img.shape != (FULL, FULL):
        img = cv2.resize(img, (FULL, FULL), interpolation=cv2.INTER_AREA)
    return img

## 4. Input channels for fine structure
- **raw**: the H-alpha image
- **CLAHE**: contrast-limited adaptive histogram equalization, which brings out faint local contrast
- **black-hat** (closing − image): responds strongly to dark structures **thinner than the kernel** (barbs, narrow filament sections) and suppresses the large-scale limb darkening

Each channel is standardized with statistics of the whole image, so training crops and full-image inference see identical inputs.

In [ ]:
CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(16, 16))
BH_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (CFG['blackhat_k'], CFG['blackhat_k']))

def channel_stack(raw, cl):
    bh = cv2.morphologyEx(np.ascontiguousarray(cl), cv2.MORPH_BLACKHAT, BH_KERNEL)
    return np.stack([raw, cl, bh]).astype(np.float32) / 255.

def stack_stats(x):
    return x.mean(axis=(1, 2)), x.std(axis=(1, 2)) + 1e-6

def normalize(x, stats):
    mean, std = stats
    return (x - mean[:, None, None]) / std[:, None, None]

raw = read_gray(samples[0]['path'])
demo = channel_stack(raw, CLAHE.apply(raw))
fig, ax = plt.subplots(1, 3, figsize=(21, 7))
for a, ch, t in zip(ax, demo, ['raw', 'CLAHE', f'black-hat (k={CFG["blackhat_k"]})']):
    a.imshow(ch, cmap='gray'); a.set_title(t); a.axis('off')
plt.show()

## 5. Split, consensus targets and caching
- Split by **unique image**, so annotations of the same image never leak across splits.
- Training target per image = **mean of all annotators' masks** (soft consensus in [0, 1]), at full 2048 resolution.
- The raw + CLAHE channels are cached (black-hat is recomputed per crop, which is cheap).

In [ ]:
C = CFG['crop']
stems = sorted(samples_by_stem)
random.Random(CFG['seed']).shuffle(stems)
val_stems = sorted(stems[:max(1, int(len(stems) * CFG['val_frac']))])
train_stems = sorted(set(stems) - set(val_stems))
val_samples = [s for st in val_stems for s in samples_by_stem[st]]
print(f'train images: {len(train_stems)}  val images: {len(val_stems)}  val annotator sets: {len(val_samples)}')

chan_cache, stat_cache, mask_cache, fg_cache = {}, {}, {}, {}
for st in tqdm(stems, desc='cache'):
    raw = read_gray(train_paths[st])
    cl = CLAHE.apply(raw)
    chan_cache[st] = np.stack([raw, cl])
    stat_cache[st] = stack_stats(channel_stack(raw, cl))
    cons = np.mean([semantic_mask(s).astype(np.float32) for s in samples_by_stem[st]], axis=0)
    mask_cache[st] = np.round(cons * 255).astype(np.uint8)
    ys, xs = np.nonzero(mask_cache[st] > 127)
    fg_cache[st] = (ys[::20], xs[::20])   # subsampled filament pixels for crop sampling

## 6. Dataset, model, loss (BCE + Dice + clDice)

In [ ]:
class CropDataset(Dataset):
    # Random 1024 crops at native resolution, biased toward filament pixels
    def __init__(self, stems_, repeats):
        self.items = [st for st in stems_ for _ in range(repeats)]
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        st = self.items[i]
        ys, xs = fg_cache[st]
        if len(ys) and random.random() < CFG['fg_crop_prob']:
            j = random.randrange(len(ys))
            cy = ys[j] + random.randint(-C // 4, C // 4)
            cx = xs[j] + random.randint(-C // 4, C // 4)
        else:
            cy, cx = random.randint(0, FULL), random.randint(0, FULL)
        y0 = int(np.clip(cy - C // 2, 0, FULL - C))
        x0 = int(np.clip(cx - C // 2, 0, FULL - C))
        raw = chan_cache[st][0, y0:y0 + C, x0:x0 + C]
        cl = chan_cache[st][1, y0:y0 + C, x0:x0 + C]
        x = normalize(channel_stack(raw, cl), stat_cache[st])
        m = mask_cache[st][y0:y0 + C, x0:x0 + C].astype(np.float32) / 255.

        k = random.randint(0, 3)
        x, m = np.rot90(x, k, axes=(1, 2)), np.rot90(m, k)
        if random.random() < 0.5:
            x, m = x[:, :, ::-1], m[:, ::-1]
        x = x * random.uniform(0.9, 1.1) + random.uniform(-0.1, 0.1)
        return torch.from_numpy(x.copy()), torch.from_numpy(m.copy())[None]

class FullDataset(Dataset):
    # Whole 2048 images (validation)
    def __init__(self, stems_):
        self.stems = stems_
    def __len__(self):
        return len(self.stems)
    def __getitem__(self, i):
        st = self.stems[i]
        x = normalize(channel_stack(*chan_cache[st]), stat_cache[st])
        m = mask_cache[st].astype(np.float32) / 255.
        return torch.from_numpy(x), torch.from_numpy(m)[None]

train_dl = DataLoader(CropDataset(train_stems, CFG['repeats']), batch_size=CFG['batch_size'], shuffle=True,
                      num_workers=CFG['num_workers'], drop_last=True, pin_memory=True,
                      persistent_workers=CFG['num_workers'] > 0)
val_dl = DataLoader(FullDataset(val_stems), batch_size=1, shuffle=False, num_workers=CFG['num_workers'],
                    persistent_workers=CFG['num_workers'] > 0)

def build_model(weights='imagenet'):
    return smp.Unet(CFG['encoder'], encoder_weights=weights, in_channels=3, classes=1,
                    decoder_attention_type='scse')

try:
    model = build_model()
except Exception as e:   # e.g. no internet for pretrained weights
    print('Pretrained weights unavailable, training from scratch:', e)
    model = build_model(None)
model = model.to(DEVICE)

In [ ]:
bce = nn.BCEWithLogitsLoss()

def dice_loss(p, target, eps=1.0):
    # Batch-level Dice: stable when some crops contain no filaments
    return 1 - (2 * (p * target).sum() + eps) / (p.sum() + target.sum() + eps)

# clDice (Shit et al., CVPR 2021): Dice between soft skeletons, so thin structures count as much as thick ones
def soft_erode(x):
    return -F.max_pool2d(-x, 3, 1, 1)

def soft_dilate(x):
    return F.max_pool2d(x, 3, 1, 1)

def soft_skel(x, iters):
    skel = F.relu(x - soft_dilate(soft_erode(x)))
    for _ in range(iters):
        x = soft_erode(x)
        delta = F.relu(x - soft_dilate(soft_erode(x)))
        skel = skel + F.relu(delta - skel * delta)
    return skel

def cldice_loss(p, target, eps=1.0):
    sp, st = soft_skel(p, CFG['skel_iters']), soft_skel(target, CFG['skel_iters'])
    tprec = ((sp * target).sum() + eps) / (sp.sum() + eps)   # predicted skeleton inside GT
    tsens = ((st * p).sum() + eps) / (st.sum() + eps)        # GT skeleton covered by prediction
    return 1 - 2 * tprec * tsens / (tprec + tsens)

def criterion(logits, target):
    logits = logits.float()
    p = torch.sigmoid(logits)
    return bce(logits, target) + dice_loss(p, target) + CFG['cldice_w'] * cldice_loss(p, target)

## 7. Train

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG['lr'], total_steps=CFG['epochs'] * len(train_dl),
                                            pct_start=0.1)
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
best_dice, history = -1, []

for epoch in range(CFG['epochs']):
    t0 = time.time()
    model.train(); tr_loss = 0
    for x, y in tqdm(train_dl, leave=False, desc=f'epoch {epoch + 1}'):
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE, enabled=USE_AMP):
            logits = model(x)
        loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        tr_loss += loss.item()

    model.eval(); va_loss, inter, union = 0, 0, 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.autocast(DEVICE, enabled=USE_AMP):
                logits = model(x)
            va_loss += criterion(logits, y).item()
            p, t = (logits > 0).float(), (y > 0.5).float()
            inter += (p * t).sum().item(); union += (p.sum() + t.sum()).item()
    dice = 2 * inter / (union + 1e-6)
    history.append((tr_loss / len(train_dl), va_loss / len(val_dl), dice))
    if dice > best_dice:
        best_dice = dice
        torch.save(model.state_dict(), 'best_model.pth')
    print(f'epoch {epoch + 1:02d} | train {history[-1][0]:.4f} | val {history[-1][1]:.4f} '
          f'| val dice {dice:.4f} | {time.time() - t0:.0f}s')

model.load_state_dict(torch.load('best_model.pth', map_location=DEVICE))
print('best val dice:', round(best_dice, 4))

In [ ]:
h = np.array(history)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h[:, 0], label='train'); ax[0].plot(h[:, 1], label='val'); ax[0].set_title('loss'); ax[0].legend()
ax[1].plot(h[:, 2]); ax[1].set_title('val pixel Dice (consensus > 0.5)')
plt.show()

## 8. Prediction, post-processing and Panoptic Quality
- **Prediction:** full 2048 image, with dihedral TTA
- **Hysteresis:** keep low-threshold regions only if they contain a high-confidence seed. Faint barb pixels attached to a filament survive, isolated noise does not, and faint gaps inside a filament are bridged.
- **Fill holes**, then **group** nearby fragments (dilation used only for labelling) and drop small components
- **PQ = SQ × RQ**: SQ = mean IoU of matches (mask accuracy), RQ = TP / (TP + 0.5 FP + 0.5 FN) (detection), match requires IoU > 0.5

In [ ]:
TTA_OPS = {1: [(0, False)], 2: [(0, False), (0, True)],
           8: [(k, f) for k in range(4) for f in (False, True)]}[CFG['tta']]

@torch.no_grad()
def predict_prob(raw):
    x = channel_stack(raw, CLAHE.apply(raw))
    x = torch.from_numpy(normalize(x, stack_stats(x)))[None].to(DEVICE)
    acc = torch.zeros((1, 1, FULL, FULL), device=DEVICE)
    for k, flip in TTA_OPS:
        t = torch.rot90(x, k, (2, 3))
        if flip:
            t = t.flip(-1)
        with torch.autocast(DEVICE, enabled=USE_AMP):
            p = torch.sigmoid(model(t)).float()
        if flip:
            p = p.flip(-1)
        acc += torch.rot90(p, -k, (2, 3))
    return (acc / len(TTA_OPS))[0, 0].cpu().numpy()

def postprocess(prob, low=0.35, high=0.6, group_k=21, min_area=400):
    # Hysteresis: low-threshold regions that contain at least one high-confidence pixel
    region = ndimage.binary_fill_holes(prob > low).astype(np.uint8)
    n, lab = cv2.connectedComponents(region, connectivity=8)
    keep = np.zeros(n, bool)
    keep[np.unique(lab[prob > high])] = True
    keep[0] = False
    binm = keep[lab].astype(np.uint8)

    grouping = binm
    if group_k > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (group_k, group_k))
        grouping = cv2.dilate(binm, kernel)
    n, lab = cv2.connectedComponents(grouping, connectivity=8)
    lab = lab * binm                                   # keep only real foreground pixels
    areas = np.bincount(lab.ravel(), minlength=n)
    return [lab == k for k in range(1, n) if areas[k] >= min_area]

def pq_stats(gt, pred):
    # Returns (sum IoU of matches, TP, FP, FN). IoU > 0.5 guarantees unique matching.
    pred_lab = np.zeros((FULL, FULL), np.int32)
    for i, p in enumerate(pred, 1):
        pred_lab[p] = i
    pred_areas = np.bincount(pred_lab.ravel(), minlength=len(pred) + 1)
    iou_sum, tp, matched = 0.0, 0, set()
    for g in gt:
        inter = np.bincount(pred_lab[g], minlength=len(pred) + 1)
        inter[0] = 0
        j = int(inter.argmax())
        if inter[j] == 0:
            continue
        iou = inter[j] / (g.sum() + pred_areas[j] - inter[j])
        if iou > 0.5:
            iou_sum += iou; tp += 1; matched.add(j)
    return iou_sum, tp, len(pred) - len(matched), len(gt) - tp

def panoptic_quality(stats):
    iou_sum, tp, fp, fn = map(sum, zip(*stats))
    sq = iou_sum / (tp + 1e-9)
    rq = tp / (tp + 0.5 * fp + 0.5 * fn + 1e-9)
    return dict(PQ=sq * rq, SQ=sq, RQ=rq, TP=tp, FP=fp, FN=fn)

In [ ]:
# Cache full-res probabilities (one per unique val image) and GT instances (one set per annotator)
val_probs = {st: predict_prob(chan_cache[st][0]) for st in tqdm(val_stems, desc='val predict')}
val_gt = [(s['stem'], instance_masks(s)) for s in tqdm(val_samples, desc='val gt')]

def evaluate(pp):
    preds = {st: postprocess(p, **pp) for st, p in val_probs.items()}
    return panoptic_quality([pq_stats(gt, preds[st]) for st, gt in val_gt])

results = []
def run(pp):
    m = evaluate(pp)
    results.append(dict(pp, **m))
    print(pp, '-> PQ={PQ:.4f} SQ={SQ:.4f} RQ={RQ:.4f} TP={TP} FP={FP} FN={FN}'.format(**m))

# Stage 1: thresholds (grouping / min area fixed)
best_pp = dict(low=0.35, high=0.6, group_k=21, min_area=400)
for low in [0.2, 0.3, 0.4, 0.5]:
    for high in [0.5, 0.65, 0.8]:
        if high >= low:
            run(dict(best_pp, low=low, high=high))
r = max(results, key=lambda d: d['PQ'])
best_pp.update(low=r['low'], high=r['high'])

# Stage 2: grouping and min area (thresholds fixed)
for group_k in [0, 21, 41, 61]:
    for min_area in [200, 400, 800, 1500]:
        run(dict(best_pp, group_k=group_k, min_area=min_area))

res = pd.DataFrame(results).sort_values('PQ', ascending=False)
r = res.iloc[0]
BEST_PP = dict(low=float(r['low']), high=float(r['high']), group_k=int(r['group_k']), min_area=int(r['min_area']))
print('Best post-processing:', BEST_PP)
print('val PQ={PQ:.4f} SQ={SQ:.4f} RQ={RQ:.4f} TP={TP} FP={FP} FN={FN}'.format(**r))
res.head(10)

In [ ]:
def color_instances(masks):
    out = np.zeros((FULL, FULL, 3), np.uint8)
    rng = np.random.default_rng(0)
    for m in masks:
        out[m] = rng.integers(60, 255, 3)
    return out

for st, gt in val_gt[:3]:
    pred = postprocess(val_probs[st], **BEST_PP)
    img = chan_cache[st][0]
    fig, ax = plt.subplots(1, 3, figsize=(21, 7))
    ax[0].imshow(img, cmap='gray'); ax[0].set_title(st)
    ax[1].imshow(img, cmap='gray'); ax[1].imshow(color_instances(gt), alpha=0.5); ax[1].set_title(f'GT ({len(gt)})')
    ax[2].imshow(img, cmap='gray'); ax[2].imshow(color_instances(pred), alpha=0.5); ax[2].set_title(f'Pred ({len(pred)})')
    for a in ax: a.axis('off')
    plt.show()

In [ ]:
# Zoom on one filament to inspect barb capture (GT outline red, prediction outline cyan)
st, gt = val_gt[0]
pred = postprocess(val_probs[st], **BEST_PP)
g = max(gt, key=lambda m: m.sum())
ys, xs = np.nonzero(g)
y0, y1 = max(ys.min() - 40, 0), min(ys.max() + 40, FULL)
x0, x1 = max(xs.min() - 40, 0), min(xs.max() + 40, FULL)
crop = cv2.cvtColor(chan_cache[st][0][y0:y1, x0:x1], cv2.COLOR_GRAY2RGB)
for masks, color in [(gt, (255, 0, 0)), (pred, (0, 255, 255))]:
    u = np.zeros((FULL, FULL), np.uint8)
    for m in masks:
        u[m] = 1
    cnts, _ = cv2.findContours(u[y0:y1, x0:x1], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(crop, cnts, -1, color, 1)
plt.figure(figsize=(12, 12)); plt.imshow(crop); plt.title(f'{st}: GT (red) vs pred (cyan)'); plt.axis('off'); plt.show()

## 9. Predict the test set and write `submission.csv`

In [ ]:
def encode_rle(m):
    return mask_utils.encode(np.asfortranarray(m.astype(np.uint8)))['counts'].decode('ascii')

rows, fallback, empty = [], [], []
for stem, path in tqdm(sorted(test_paths.items()), desc='test'):
    prob = predict_prob(read_gray(path))
    instances = postprocess(prob, **BEST_PP)
    if not instances:   # relax size filter before giving up on an image
        instances = postprocess(prob, **dict(BEST_PP, min_area=BEST_PP['min_area'] // 4))
        fallback.append(stem)
    if not instances:
        empty.append(stem)
    for i, m in enumerate(instances, 1):
        rows.append(dict(filament_id=f'{stem}_{i}', segmentation_rle=encode_rle(m)))

sub = pd.DataFrame(rows, columns=['filament_id', 'segmentation_rle'])
sub.to_csv('submission.csv', index=False)
print(f'{len(sub)} filaments over {len(test_paths)} test images -> submission.csv')
print(f'fallback used on {len(fallback)} images, still empty: {empty}')
sub.head()

In [ ]:
# Sanity check: decode one RLE back and overlay it on its image
r = sub.iloc[0]
m = mask_utils.decode({'size': [FULL, FULL], 'counts': r.segmentation_rle.encode('ascii')})
stem = r.filament_id.rsplit('_', 1)[0]
plt.figure(figsize=(7, 7))
plt.imshow(read_gray(test_paths[stem]), cmap='gray')
plt.imshow(np.ma.masked_where(m == 0, m), cmap='autumn', alpha=0.6)
plt.title(r.filament_id); plt.axis('off'); plt.show()